Lab 5: Fintech Support Ticket Routing

## Business Context
A digital payments and lending platform receives customer queries across categories such as **Payments/UPI**, **KYC/Account**, and **Loan/EMI**.
Routing tickets quickly helps urgent payment and transaction issues reach specialized support agents faster.

### What You Will Practice
- Converting text into numerical features with `TfidfVectorizer`
- Training `MultinomialNB` and testing **Laplace smoothing (`alpha`)**
- Inspecting predicted probabilities (`predict_proba`) and confusion matrices

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

base_tickets = [
    ('UPI payment debited from bank account but merchant did not receive money', 'Payments_UPI'),
    ('QR scan failed with transaction timeout error during checkout', 'Payments_UPI'),
    ('Money deducted twice for one UPI transfer please refund', 'Payments_UPI'),
    ('Bank server down while sending instant payment via UPI ID', 'Payments_UPI'),
    ('Autopay mandate failed for recurring bill payment today', 'Payments_UPI'),
    ('Transaction pending status for two hours need payment confirmation', 'Payments_UPI'),
    ('Refund not credited back after failed card and UPI checkout', 'Payments_UPI'),
    ('Unable to link debit card for instant wallet top up', 'Payments_UPI'),
    ('PAN card verification pending for my KYC profile update', 'KYC_Account'),
    ('Aadhaar OTP not arriving to complete video KYC onboarding', 'KYC_Account'),
    ('Account temporarily frozen after multiple wrong PIN attempts', 'KYC_Account'),
    ('Need to update registered mobile number linked to my profile', 'KYC_Account'),
    ('Address proof document rejected during KYC re-verification', 'KYC_Account'),
    ('Cannot log in to app after changing phone device', 'KYC_Account'),
    ('Profile name mismatch between PAN and bank details', 'KYC_Account'),
    ('Requesting account closure and data deletion confirmation', 'KYC_Account'),
    ('When will my monthly EMI auto debit happen for personal loan', 'Loan_EMI'),
    ('Want to foreclose my loan early and check foreclosure charges', 'Loan_EMI'),
    ('Interest rate and repayment schedule details for approved credit line', 'Loan_EMI'),
    ('EMI bounce fee applied even though balance was sufficient', 'Loan_EMI'),
    ('How can I increase my sanctioned loan limit after six timely EMIs', 'Loan_EMI'),
    ('Need NOC certificate after paying off final loan installment', 'Loan_EMI'),
    ('Due date extension request for upcoming EMI payment this month', 'Loan_EMI'),
    ('Disbursal status for my personal loan application after agreement signing', 'Loan_EMI'),
]

rng = np.random.default_rng(42)
prefixes = ['', 'Urgent: ', 'Hi team, ', 'Please help - ', 'Issue since morning: ']
suffixes = ['', ' please check asap', ' ticket raised earlier', ' need help urgently', '']
rows = []
for _ in range(4):
    for text, label in base_tickets:
        p = rng.choice(prefixes)
        s = rng.choice(suffixes)
        rows.append({'ticket_text': f'{p}{text}{s}'.strip(), 'category': label})

df_tickets = pd.DataFrame(rows)
print('Total tickets:', len(df_tickets))
print(df_tickets['category'].value_counts().to_string())

Total tickets: 96
category
Payments_UPI    32
KYC_Account     32
Loan_EMI        32


## Step 1: Train-Test Split & TF-IDF Vectorization
Algorithm assumes features are conditionally independent given the class. Even though words in real support messages aren't strictly independent, `MultinomialNB` is fast and strong on text classification.

## Step 2: Laplace Smoothing (`alpha`) & Evaluation
Laplace smoothing (`alpha > 0`) prevents zero probabilities when a word in a test ticket was unseen in a given class during training.

## Step 3: Class Probabilities on Sample Tickets

In [2]:
sample_msgs = [
    'UPI amount debited twice during QR payment, refund not received',
    'Aadhaar OTP not coming for video KYC verification',
    'Want to check personal loan EMI due date and foreclosure charges'
]
probs = model.predict_proba(vectorizer.transform(sample_msgs))
prob_df = pd.DataFrame(np.round(probs, 3), columns=model.classes_)
prob_df.insert(0, 'Message', sample_msgs)
print(prob_df.to_string(index=False))

NameError: name 'model' is not defined

## Practice Exercises
1. **Smoothing extremes:** Test `alpha=1e-6` and `alpha=10.0`. How does accuracy change?
2. **Count vs. TF-IDF:** Swap `TfidfVectorizer` for `CountVectorizer` and compare classification metrics.
3. **Misclassification inspection:** List any test messages where `y_pred != y_test` and inspect their predicted probabilities.